# Kaggle CUDA device check

Prints the active PyTorch CUDA device information and writes a small result file for download from Kaggle outputs.

In [ ]:
import json
import platform
from pathlib import Path

import torch

cuda_available = torch.cuda.is_available()
device_count = torch.cuda.device_count() if cuda_available else 0
current_device = torch.cuda.current_device() if cuda_available else None

device_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "cuda_available": cuda_available,
    "torch_cuda_version": torch.version.cuda,
    "device_count": device_count,
    "current_device": current_device,
    "devices": [],
}

if cuda_available:
    for index in range(device_count):
        props = torch.cuda.get_device_properties(index)
        device_info["devices"].append(
            {
                "index": index,
                "name": torch.cuda.get_device_name(index),
                "capability": f"{props.major}.{props.minor}",
                "total_memory_gb": round(props.total_memory / 1024**3, 2),
                "multi_processor_count": props.multi_processor_count,
            }
        )

print("CUDA available:", cuda_available)
print("Torch CUDA version:", torch.version.cuda)
print("Device count:", device_count)
if cuda_available:
    print("Current CUDA device:", current_device)
    print("Active GPU:", torch.cuda.get_device_name(current_device))
else:
    print("Active GPU: none")

output_dir = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd().parent / "ouptut"
output_dir.mkdir(parents=True, exist_ok=True)

json_path = output_dir / "gpu_check.json"
text_path = output_dir / "gpu_check.txt"

json_path.write_text(json.dumps(device_info, indent=2), encoding="utf-8")
text_path.write_text(
    "\n".join(
        [
            f"cuda_available={cuda_available}",
            f"torch_cuda_version={torch.version.cuda}",
            f"device_count={device_count}",
            f"active_gpu={torch.cuda.get_device_name(current_device) if cuda_available else 'none'}",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print("Wrote:", json_path)
print("Wrote:", text_path)
